## Step 5: The TESS Model Factory (1D CNN vs. CNN+LSTM — This Time For Real)

The grand finale of the TESS pipeline — and unlike its Kepler cousin (`01_dd_CNN_LSTM.ipynb`), the "LSTM" option here isn't a decoy: it's a genuinely different architecture that actually gets trained. This notebook takes the labeled TESS IDs, downloads (or synthesizes!) their light curves as fixed-length 1D sequences, and trains a real choice between a CNN and a CNN+LSTM hybrid.

**Key structural difference from the Kepler version:** instead of turning data into 2D "images" (256×256 pixel grids), this notebook works with **1D time-series sequences** of length `TARGET_LENGTH = 1000` — which makes a lot more physical sense for light curve data, since a light curve genuinely *is* a 1D sequence over time, not a 2D image.

**The Algorithm:**

1. **`load_ids()`**: a flexible loader that transparently handles both `.csv` and `.xlsx`/`.xls` files (via `openpyxl`), always pulling from the first column and returning IDs as strings.

2. **`download_tess_data()`**: fetches and prepares one star's light curve for the model:
   - Downloads via `lightkurve`; if multiple sectors come back as a `LightCurveCollection`, it **stitches them together** into one continuous light curve — a nice touch that Notebooks 2 and 4 don't do, meaning multi-sector targets get their full observational baseline used rather than just one sector.
   - Drops NaNs, then flattens with `flatten(window_length=301, break_tolerance=0.5)` — `lightkurve`'s built-in detrending, same family of technique as everywhere else in both pipelines.
   - **Resamples to a fixed length** via `np.interp()`, mapping whatever the light curve's original length was onto exactly `TARGET_LENGTH=1000` points. This is the key trick that lets stars with wildly different numbers of observations (different missions, different numbers of sectors, different cadences) all become the same-shaped input the neural network expects.
   - Z-score normalizes the resampled flux (with a tiny `1e-8` epsilon added to the denominator, guarding against division by zero for a perfectly flat/constant light curve).
   - Returns `None` on any failure, signaling to the caller that this star needs a fallback.

3. **`synthetic_curve()`**: this is the standout feature of this notebook — when a real download fails, instead of just skipping the star, it **generates a fake light curve** as a stand-in:
   - Seeds NumPy's RNG deterministically from the TIC ID (`np.random.seed(int(tess_id) % 2**32)`) so the same "fake star" is reproducible across runs.
   - Generates baseline Gaussian noise (`mean=1.0, std=0.01`) to mimic quiet stellar flux.
   - **If the star's true label is positive**, it artificially injects a shallow dip between indices 400–600 — literally faking a transit signal — so the synthetic data still matches its assigned label instead of being pure noise mislabeled as "planet."
   - This is a pragmatic (if slightly ethically-gray) way to keep training batches full-sized even when some downloads fail, though it's worth flagging clearly in any writeup: **synthetic data mixed into real training data can inflate apparent accuracy** if the model learns to distinguish "real messy data" from "suspiciously clean synthetic data" rather than genuine transit shape.

4. **Two real model architectures, this time both actually different:**
   - **`build_cnn()`**: `Conv1D(32) → MaxPool → Conv1D(64) → MaxPool → Flatten → Dense(64) → Dropout(0.5) → Dense(1, sigmoid)` — a straightforward 1D convolutional classifier over the flux sequence.
   - **`build_cnn_lstm()`**: identical convolutional front-end, but instead of flattening after the conv/pool stack, it feeds the extracted feature maps into an **`LSTM(64, return_sequences=False)`** layer before the final dense/dropout/output layers. This lets the model learn temporal dependencies across the sequence (the CNN extracts local shape features like "dip here", the LSTM can reason about how those features relate to each other in sequence) — a legitimately more sophisticated approach than Kepler Notebook 4's "LSTM" that quietly wasn't one.

5. **`train_and_evaluate()`**: runs proper **Stratified K-Fold cross-validation** (`StratifiedKFold`, preserving the positive/negative class ratio in every fold, which matters a lot for an inherently imbalanced problem like transit detection) with `EarlyStopping(patience=5, restore_best_weights=True)` to avoid overfitting/wasting epochs once validation performance plateaus. Reports the average best validation accuracy across all folds.

6. **`main()`** ties everything together:
   - Loads labeled ID lists from `Negative_Transit.csv`, `Positive_Transit.csv`, `Skipped_Transit.csv` (from Notebook 4).
   - Interactively lets the user decide how many positive/uncertain/negative examples to include — same "build your own dataset composition" pattern as the Kepler version.
   - Lets the user pick **CNN vs. CNN+LSTM** for real this time.
   - Downloads (or synthesizes, on failure) a flux sequence for every selected ID, assembling `X` (shape: `n_samples × 1000 × 1`) and `y`.
   - **Saves the raw dataset** (`1d_X_data.npy`, `1d_y_labels.npy`) before any training happens — so re-running the training step later doesn't require re-downloading everything.
   - Dynamically picks the number of CV folds based on the smaller class's size (`min(5, min_class)`, floored at 2) — a sensible guard against asking for more folds than your minority class can support.
   - Runs cross-validated training for a performance estimate, then **trains one final model on 100% of the data** (no held-out validation split) — a common and reasonable pattern once you're satisfied with the CV results and want the best possible model to actually deploy.
   - Saves the final model as `best_1d_trained_model.keras`.

7. **A short evaluation tail** at the end of the notebook: reloads the saved `X`/`y` arrays and the trained model, and reports training accuracy/loss on the full dataset — a quick "did this actually learn something" gut-check (note: this is *training* accuracy, not held-out validation accuracy, so it should be read as a sanity check rather than a true performance estimate).

**Output:** `1d_X_data.npy` / `1d_y_labels.npy` (the processed dataset), `best_1d_trained_model.keras` (the final trained classifier) — the end product of the entire TESS pipeline.

*TL;DR: raw flux sequences in (real or, when necessary, tastefully faked), a genuine CNN-vs-CNN+LSTM decision out — and this time, when it says LSTM, it actually means it.*

In [1]:
# pip install pandas numpy scikit-learn tensorflow lightkurve openpyxl

In [2]:
import pandas as pd
import numpy as np
import lightkurve as lk
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import warnings

warnings.filterwarnings('ignore')

In [3]:
TARGET_LENGTH = 1000

In [4]:
def load_ids(file_path):
    if file_path.lower().endswith('.csv'):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path, engine='openpyxl')
    return df.iloc[:, 0].dropna().astype(str).tolist()

In [5]:
def download_tess_data(tess_id):
    try:
        lc = lk.search_lightcurve(f"TIC {tess_id}", mission="TESS").download()
        
        if isinstance(lc, lk.LightCurveCollection):
            lc = lc.stitch()
            
        lc = lc[~np.isnan(lc.flux)]
        
        if len(lc.flux) == 0:
            return None
        lc_flat = lc.flatten(window_length=301, break_tolerance=0.5)
        
        flux = lc_flat.flux.value
        
        x_old = np.linspace(0, 1, len(flux))
        x_new = np.linspace(0, 1, TARGET_LENGTH)
        flux_interp = np.interp(x_new, x_old, flux)
        
        flux_interp = (flux_interp - np.mean(flux_interp)) / (np.std(flux_interp) + 1e-8)
        return flux_interp
        
    except Exception as e:
        print(f"   Download error for TIC {tess_id}: {e}")
        return None

In [6]:
def synthetic_curve(tess_id, positive_class_ids):
    np.random.seed(int(tess_id) % 2**32)
    flux = np.random.normal(1.0, 0.01, TARGET_LENGTH)
    
    if str(tess_id) in positive_class_ids:
        dip_start, dip_end = 400, 600
        flux[dip_start:dip_end] -= 0.05 * np.random.rand(dip_end - dip_start)
        
    flux = (flux - np.mean(flux)) / (np.std(flux) + 1e-8)
    return flux

In [7]:
def build_cnn(input_shape):
    model = Sequential([
        Conv1D(32, 3, activation='relu', input_shape=input_shape),
        MaxPooling1D(2), Conv1D(64, 3, activation='relu'),
        MaxPooling1D(2), Flatten(), Dense(64, activation='relu'),
        Dropout(0.5), Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [8]:
def build_cnn_lstm(input_shape):
    model = Sequential([
        Conv1D(32, 3, activation='relu', input_shape=input_shape),
        MaxPooling1D(2), Conv1D(64, 3, activation='relu'),
        MaxPooling1D(2), LSTM(64, return_sequences=False),
        Dropout(0.5), Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [9]:
def train_and_evaluate(model_fn, X, y, n_folds, epochs=30, batch_size=32):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    val_accs = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"\nFold {fold}/{n_folds}")
        
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        model = model_fn((X.shape[1], X.shape[2]))
        
        history = model.fit(
            X_train, y_train, validation_data=(X_val, y_val),
            epochs=epochs, batch_size=batch_size,
            verbose=1, callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
        )
        best_val_acc = max(history.history['val_accuracy'])
        val_accs.append(best_val_acc)
        
    avg_acc = np.mean(val_accs)
    
    print(f"Average validation accuracy: {avg_acc:.4f}")
    return avg_acc

In [10]:
def main():
    negative_ids = load_ids('Negative_Transit.csv')
    positive_ids = load_ids('Positive_Transit.csv')
    uncertain_ids = load_ids('Skipped_Transit.csv')

    print(f"Negative: {len(negative_ids)}, Positive: {len(positive_ids)}, Uncertain: {len(uncertain_ids)}")

    use_both = input(f"Positive={len(positive_ids)}, Uncertain={len(uncertain_ids)}. Use both? (y/n): ").strip().lower()
    
    if use_both in ('y','yes'):
        pos_use = int(input(f"How many positive? (max {len(positive_ids)}): "))
        pos_use = max(0, min(pos_use, len(positive_ids)))
        unc_use = int(input(f"How many uncertain? (max {len(uncertain_ids)}): "))
        unc_use = max(0, min(unc_use, len(uncertain_ids)))
        positive_class_ids = positive_ids[:pos_use] + uncertain_ids[:unc_use]
    else:
        positive_class_ids = positive_ids[:]
        print(f"Using all positive ({len(positive_class_ids)}) and no uncertain.")

    neg_use = int(input(f"How many negative? (max {len(negative_ids)}): "))
    neg_use = max(0, min(neg_use, len(negative_ids)))
    selected_negative = negative_ids[:neg_use]

    all_ids = positive_class_ids + selected_negative
    all_labels = [1]*len(positive_class_ids) + [0]*len(selected_negative)
    print(f"Total: {len(all_ids)} (Positive: {len(positive_class_ids)}, Negative: {len(selected_negative)})")

    print('Which model do you wish to use:')
    print('    1. CNN')
    print('    2. CNN_LSTM')
    
    choice = int(input('Choose your model (1 or 2): '))
    
    if choice == 1:
        model_choice = 'cnn'
    else:
        model_choice = 'cnn_lstm'
        
    print(f'Your selected model is {model_choice.upper()}')

    print("\nDownloading light curves...")
    
    X, y = [], []
    
    for idx, (tid, label) in enumerate(zip(all_ids, all_labels), 1):
        print(f"Processing ID {tid} ({idx}/{len(all_ids)})...")
        flux = download_tess_data(tid)
        if flux is None:
            print(f"   Using synthetic curve for {tid}")
            flux = synthetic_curve(tid, positive_class_ids)
        X.append(flux)
        y.append(label)

    X = np.array(X).reshape(-1, TARGET_LENGTH, 1)
    y = np.array(y)
    print(f"Data shape: {X.shape}, labels: {y.shape}")

    np.save('1d_X_data.npy', X)
    np.save('1d_y_labels.npy', y)
    print("Data saved to X_data.npy and y_labels.npy")

    min_class = min(np.sum(y==0), np.sum(y==1))
    n_folds = min(5, min_class)
    if n_folds < 2:
        n_folds = 2
    print(f"\nUsing {n_folds}-fold CV.")

    model_fn = build_cnn if model_choice == 'cnn' else build_cnn_lstm
    print(f"\nTraining {model_choice.upper()} with CV...")
    avg_acc = train_and_evaluate(model_fn, X, y, n_folds, epochs=30)

    print("\nTraining final model on all data...")
    
    final_model = model_fn((X.shape[1], X.shape[2]))
    history = final_model.fit(X, y, epochs=30, batch_size=32, verbose=1)
    final_model.save('best_1d_trained_model.keras')
    
    print("\nModel saved as 'best_trained_model.keras'.")
    print("Done!")

In [11]:
if __name__ == "__main__":
    main()

Negative: 27, Positive: 3, Uncertain: 12


Positive=3, Uncertain=12. Use both? (y/n):  y
How many positive? (max 3):  3
How many uncertain? (max 12):  12
How many negative? (max 27):  15


Total: 30 (Positive: 15, Negative: 15)
Which model do you wish to use:
    1. CNN
    2. CNN_LSTM


Choose your model (1 or 2):  2


Your selected model is CNN_LSTM

Processing ID 209459275 (1/30)...
Processing ID 92352620 (2/30)...
Processing ID 149603524 (3/30)...
Processing ID 441462736 (4/30)...
Processing ID 410214986 (5/30)...
Processing ID 224225541 (6/30)...
Processing ID 200723869 (7/30)...


Processing ID 259377017 (8/30)...


Processing ID 94986319 (9/30)...
Processing ID 31374837 (10/30)...
Processing ID 70899085 (11/30)...
Processing ID 257605131 (12/30)...
Processing ID 98796344 (13/30)...
Processing ID 200322593 (14/30)...
Processing ID 101011575 (15/30)...
Processing ID 183985250 (16/30)...
Processing ID 350618622 (17/30)...
Processing ID 55650590 (18/30)...
Processing ID 150098860 (19/30)...
Processing ID 305048087 (20/30)...
Processing ID 220479565 (21/30)...


Processing ID 29781292 (22/30)...
Processing ID 178155732 (23/30)...
Processing ID 120896927 (24/30)...


Processing ID 452866790 (25/30)...
Processing ID 134200185 (26/30)...
Processing ID 453211454 (27/30)...
Processing ID 218795833 (28/30)...
Processing ID 387690507 (29/30)...
Processing ID 144700903 (30/30)...
Data shape: (30, 1000, 1), labels: (30,)
Data saved to X_data.npy and y_labels.npy

Using 5-fold CV.

Training CNN_LSTM with CV...

Fold 1/5
Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.3333 - loss: 0.7019 - val_accuracy: 0.5000 - val_loss: 0.7007
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.4583 - loss: 0.6839 - val_accuracy: 0.5000 - val_loss: 0.7010
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.5833 - loss: 0.6911 - val_accuracy: 0.5000 - val_loss: 0.7019
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.3750 - loss: 0.6940 - val_accuracy: 0.5000 - val_loss: 0.7026
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.4583 - loss: 0.6933 - val_accuracy: 0.5000 - val_loss: 0.7026
Epoch 6/30
1/1 ━━━━━━━

In [12]:
X = np.load('1d_X_data.npy')
y = np.load('1d_y_labels.npy')
print(f"Loaded data: X.shape = {X.shape}, y.shape = {y.shape}")

Loaded data: X.shape = (30, 1000, 1), y.shape = (30,)


In [13]:
model = load_model('best_1d_trained_model.keras')

In [14]:
loss, acc = model.evaluate(X, y, verbose=0)
print(f"Training accuracy: {acc:.4f}")
print(f"   Training loss: {loss:.4f}")

Training accuracy: 0.7667
   Training loss: 0.4398
